# 19 — PyTorch → Keras DenseNet121 Parity Test (ONE checkpoint)

Mục tiêu: kiểm tra xem checkpoint PyTorch hiện tại có thể chuyển sang Keras **mà không làm thay đổi output** hay không.

Checkpoint thử nghiệm:
- grayscale DenseNet121
- seed 2026
- fold 0
- `D:\medicine\cv_outputs\densenet121_224_gray_b8a2\fold0_best.pt`

Chiến lược:
1. Dựng DenseNet121 bằng các layer chuẩn của `tf.keras`, bám đúng kiến trúc torchvision.
2. Map Conv / BatchNorm / classifier weights từ PyTorch sang Keras.
3. Lấy cùng một batch ảnh đã preprocess bằng pipeline PyTorch.
4. PyTorch nhận NCHW; Keras nhận chính tensor đó sau transpose thành NHWC.
5. So logits, probability và predicted class.
6. Save thành `.keras`, load lại bằng `tf.keras.models.load_model()`, kiểm tra parity lần nữa.

**Pass criteria ban đầu**
- class agreement = 100%
- max probability difference <= 1e-4
- reload `.keras` không làm thay đổi output

Nếu fail: dừng conversion, không convert ensemble.


In [ ]:
print('>>> CELL 1 START: environment', flush=True)
import os,sys,json,gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torchvision import models,transforms

try:
    import tensorflow as tf
except Exception as e:
    raise RuntimeError(
        'TensorFlow is required for this conversion test. '
        'Do not retrain anything yet; install/enable an allowed TensorFlow environment first. '
        f'Original import error: {e}'
    )

ROOT=Path(r'D:\medicine')
CHECKPOINT=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2'/'fold0_best.pt'
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'conversion_outputs'/'pytorch_to_keras_parity'
OUT.mkdir(parents=True,exist_ok=True)

KERAS_PATH=OUT/'densenet121_seed2026_fold0.keras'
N_SAMPLES=64
IMAGE_SIZE=224

print('Python:',sys.version.split()[0],flush=True)
print('PyTorch:',torch.__version__,flush=True)
print('TensorFlow:',tf.__version__,flush=True)
print('Checkpoint:',CHECKPOINT,flush=True)

assert CHECKPOINT.exists(),f'Missing checkpoint: {CHECKPOINT}'
assert MANIFEST.exists(),f'Missing manifest: {MANIFEST}'


In [ ]:
print('>>> CELL 2 START: load PyTorch checkpoint', flush=True)

ck=torch.load(CHECKPOINT,map_location='cpu')

pt_model=models.densenet121(weights=None)
pt_model.classifier=nn.Linear(pt_model.classifier.in_features,2)
pt_model.load_state_dict(ck['state_dict'],strict=True)
pt_model.eval()

state=pt_model.state_dict()

print('State tensors:',len(state),flush=True)
print('Classifier:',tuple(state['classifier.weight'].shape),tuple(state['classifier.bias'].shape),flush=True)
print('Best epoch:',ck.get('best_epoch','unknown'),flush=True)


In [ ]:
print('>>> CELL 3 START: build exact Keras DenseNet121', flush=True)

L=tf.keras.layers

def bn(x,name,layer_map):
    layer=L.BatchNormalization(
        axis=-1,
        momentum=0.1,       # inference parity does not depend on momentum
        epsilon=1e-5,      # match torch.nn.BatchNorm2d default
        center=True,
        scale=True,
        name=name.replace('.','_')
    )
    y=layer(x)
    layer_map[name]=layer
    return y

def conv(x,filters,kernel,strides=1,name=None,layer_map=None):
    layer=L.Conv2D(
        filters,
        kernel_size=kernel,
        strides=strides,
        padding='valid',
        use_bias=False,
        data_format='channels_last',
        name=name.replace('.','_')
    )
    y=layer(x)
    layer_map[name]=layer
    return y

def build_torchvision_densenet121_keras(input_shape=(224,224,3),classes=2):
    layer_map={}
    inp=tf.keras.Input(shape=input_shape,dtype=tf.float32,name='image')

    # torchvision: Conv2d(7,stride=2,padding=3)
    x=L.ZeroPadding2D(padding=3,name='features_conv0_pad')(inp)
    x=conv(x,64,7,strides=2,name='features.conv0',layer_map=layer_map)
    x=bn(x,'features.norm0',layer_map)
    x=L.ReLU(name='features_relu0')(x)

    # torchvision: MaxPool2d(3,stride=2,padding=1)
    x=L.ZeroPadding2D(padding=1,name='features_pool0_pad')(x)
    x=L.MaxPooling2D(pool_size=3,strides=2,padding='valid',name='features_pool0')(x)

    block_config=(6,12,24,16)
    growth_rate=32
    bn_size=4
    num_features=64

    for block_idx,num_layers in enumerate(block_config, start=1):
        features=[x]

        for layer_idx in range(1,num_layers+1):
            cat=features[0] if len(features)==1 else L.Concatenate(
                axis=-1,
                name=f'features_denseblock{block_idx}_denselayer{layer_idx}_input_concat'
            )(features)

            prefix=f'features.denseblock{block_idx}.denselayer{layer_idx}'

            y=bn(cat,prefix+'.norm1',layer_map)
            y=L.ReLU(name=(prefix+'.relu1').replace('.','_'))(y)
            y=conv(
                y,
                bn_size*growth_rate,
                1,
                name=prefix+'.conv1',
                layer_map=layer_map
            )

            y=bn(y,prefix+'.norm2',layer_map)
            y=L.ReLU(name=(prefix+'.relu2').replace('.','_'))(y)
            y=L.ZeroPadding2D(
                padding=1,
                name=(prefix+'.conv2_pad').replace('.','_')
            )(y)
            y=conv(
                y,
                growth_rate,
                3,
                name=prefix+'.conv2',
                layer_map=layer_map
            )

            features.append(y)

        x=L.Concatenate(
            axis=-1,
            name=f'features_denseblock{block_idx}_output_concat'
        )(features)
        num_features += num_layers*growth_rate

        if block_idx != 4:
            prefix=f'features.transition{block_idx}'
            x=bn(x,prefix+'.norm',layer_map)
            x=L.ReLU(name=(prefix+'.relu').replace('.','_'))(x)
            x=conv(
                x,
                num_features//2,
                1,
                name=prefix+'.conv',
                layer_map=layer_map
            )
            x=L.AveragePooling2D(
                pool_size=2,
                strides=2,
                padding='valid',
                name=(prefix+'.pool').replace('.','_')
            )(x)
            num_features//=2

    x=bn(x,'features.norm5',layer_map)
    x=L.ReLU(name='features_relu5')(x)
    x=L.GlobalAveragePooling2D(name='adaptive_avg_pool')(x)

    classifier=L.Dense(classes,use_bias=True,name='classifier')
    logits=classifier(x)
    layer_map['classifier']=classifier

    model=tf.keras.Model(inp,logits,name='torchvision_densenet121_exact')
    return model,layer_map

keras_model,layer_map=build_torchvision_densenet121_keras()

print('Keras params:',keras_model.count_params(),flush=True)
print('Mapped weighted layers:',len(layer_map),flush=True)


In [ ]:
print('>>> CELL 4 START: map PyTorch weights -> Keras', flush=True)

def np_pt(key):
    return state[key].detach().cpu().numpy()

mapped=[]

for prefix,layer in layer_map.items():
    if prefix=='classifier':
        # PyTorch Linear: [out,in] -> Keras Dense: [in,out]
        kernel=np_pt('classifier.weight').T
        bias=np_pt('classifier.bias')
        layer.set_weights([kernel,bias])
        mapped.extend(['classifier.weight','classifier.bias'])
        continue

    if isinstance(layer,tf.keras.layers.Conv2D):
        # PyTorch Conv2d: [out,in,h,w] -> Keras Conv2D: [h,w,in,out]
        w=np_pt(prefix+'.weight')
        w=np.transpose(w,(2,3,1,0))
        layer.set_weights([w])
        mapped.append(prefix+'.weight')
        continue

    if isinstance(layer,tf.keras.layers.BatchNormalization):
        gamma=np_pt(prefix+'.weight')
        beta=np_pt(prefix+'.bias')
        mean=np_pt(prefix+'.running_mean')
        var=np_pt(prefix+'.running_var')
        layer.set_weights([gamma,beta,mean,var])
        mapped.extend([
            prefix+'.weight',
            prefix+'.bias',
            prefix+'.running_mean',
            prefix+'.running_var'
        ])
        continue

    raise TypeError(f'Unexpected mapped layer type: {prefix} -> {type(layer)}')

ignored=[k for k in state.keys() if k.endswith('num_batches_tracked')]
expected=[k for k in state.keys() if k not in ignored]

missing=sorted(set(expected)-set(mapped))
extra=sorted(set(mapped)-set(expected))

print('Mapped tensors:',len(mapped),flush=True)
print('Ignored num_batches_tracked:',len(ignored),flush=True)
print('Missing:',missing[:20],flush=True)
print('Extra:',extra[:20],flush=True)

assert not missing,'Some PyTorch tensors were not mapped.'
assert not extra,'Unexpected mapped tensor names.'
print('WEIGHT MAPPING: PASS',flush=True)


In [ ]:
print('>>> CELL 5 START: build identical preprocessed sample batch', flush=True)

manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy()
sample_df=dev[dev.cv_fold==0].head(N_SAMPLES).copy().reset_index(drop=True)

assert len(sample_df)==N_SAMPLES

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        s=max(w,h)
        l=(s-w)//2;r=s-w-l
        t=(s-h)//2;b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pt_tf=transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

xs=[]
names=[]

for _,r in sample_df.iterrows():
    with Image.open(r.path) as im:
        x=pt_tf(im.convert('L').convert('RGB'))
    xs.append(x)
    names.append(r.filename)

x_pt=torch.stack(xs,dim=0)
x_keras=np.transpose(x_pt.numpy(),(0,2,3,1)).astype(np.float32)

print('PyTorch input:',tuple(x_pt.shape),flush=True)
print('Keras input:',x_keras.shape,flush=True)
print('Sample count:',len(names),flush=True)


In [ ]:
print('>>> CELL 6 START: parity BEFORE save', flush=True)

with torch.no_grad():
    logits_pt=pt_model(x_pt).cpu().numpy()
    probs_pt=torch.softmax(torch.from_numpy(logits_pt),dim=1).numpy()[:,1]

logits_k=keras_model(x_keras,training=False).numpy()
probs_k=tf.nn.softmax(logits_k,axis=1).numpy()[:,1]

logit_abs=np.abs(logits_pt-logits_k)
prob_abs=np.abs(probs_pt-probs_k)

pred_pt=(probs_pt>=0.5).astype(int)
pred_k=(probs_k>=0.5).astype(int)

report={
    'n_samples':int(N_SAMPLES),
    'max_abs_logit_diff':float(logit_abs.max()),
    'mean_abs_logit_diff':float(logit_abs.mean()),
    'max_abs_probability_diff':float(prob_abs.max()),
    'mean_abs_probability_diff':float(prob_abs.mean()),
    'class_agreement_at_0_5':float((pred_pt==pred_k).mean()),
}

print(json.dumps(report,indent=2),flush=True)

compare=pd.DataFrame({
    'filename':names,
    'pytorch_p_pneumonia':probs_pt,
    'keras_p_pneumonia':probs_k,
    'abs_diff':prob_abs,
    'pytorch_pred':pred_pt,
    'keras_pred':pred_k,
}).sort_values('abs_diff',ascending=False)

compare.to_csv(OUT/'parity_before_save.csv',index=False)
display(compare.head(20))

PASS_BEFORE=(
    report['class_agreement_at_0_5']==1.0
    and report['max_abs_probability_diff']<=1e-4
)

print('PARITY BEFORE SAVE:', 'PASS' if PASS_BEFORE else 'FAIL',flush=True)


In [ ]:
print('>>> CELL 7 START: save .keras + tf.keras load_model reload test', flush=True)

if not PASS_BEFORE:
    raise RuntimeError(
        'Parity failed before saving. STOP HERE. Do not convert the ensemble.'
    )

keras_model.save(KERAS_PATH)
print('Saved:',KERAS_PATH,flush=True)

reloaded=tf.keras.models.load_model(KERAS_PATH,compile=False)
logits_reload=reloaded(x_keras,training=False).numpy()
probs_reload=tf.nn.softmax(logits_reload,axis=1).numpy()[:,1]

reload_vs_pt=np.abs(probs_pt-probs_reload)
reload_vs_k=np.abs(probs_k-probs_reload)

reload_report={
    'keras_path':str(KERAS_PATH),
    'max_abs_probability_diff_pytorch_vs_reloaded':float(reload_vs_pt.max()),
    'mean_abs_probability_diff_pytorch_vs_reloaded':float(reload_vs_pt.mean()),
    'max_abs_probability_diff_keras_before_vs_reloaded':float(reload_vs_k.max()),
    'class_agreement_pytorch_vs_reloaded_at_0_5':float(
        ((probs_pt>=0.5)==(probs_reload>=0.5)).mean()
    ),
}

PASS_RELOAD=(
    reload_report['class_agreement_pytorch_vs_reloaded_at_0_5']==1.0
    and reload_report['max_abs_probability_diff_pytorch_vs_reloaded']<=1e-4
)

print(json.dumps(reload_report,indent=2),flush=True)
print('KERAS RELOAD PARITY:', 'PASS' if PASS_RELOAD else 'FAIL',flush=True)


In [ ]:
print('>>> CELL 8 START: final conversion decision', flush=True)

decision={
    'source_checkpoint':str(CHECKPOINT),
    'keras_file':str(KERAS_PATH),
    'before_save':report,
    'after_reload':reload_report if 'reload_report' in globals() else None,
    'pass_before_save':bool(PASS_BEFORE),
    'pass_after_reload':bool(PASS_RELOAD) if 'PASS_RELOAD' in globals() else False,
    'safe_to_attempt_full_ensemble_conversion':bool(
        PASS_BEFORE and ('PASS_RELOAD' in globals()) and PASS_RELOAD
    )
}

with open(OUT/'conversion_parity_summary.json','w',encoding='utf-8') as f:
    json.dump(decision,f,indent=2)

print(json.dumps(decision,indent=2),flush=True)

if decision['safe_to_attempt_full_ensemble_conversion']:
    print('\nRESULT: PASS — one checkpoint can be reproduced in Keras with parity.',flush=True)
else:
    print('\nRESULT: FAIL — do NOT convert the remaining checkpoints.',flush=True)


## Sau khi chạy

Gửi nội dung:

```powershell
Get-Content .\conversion_outputs\pytorch_to_keras_parity\conversion_parity_summary.json
```

Nếu `safe_to_attempt_full_ensemble_conversion = true`, bước tiếp theo mới là convert đủ 10 checkpoint và đóng gói ensemble thành một file `.keras`.

Nếu false, dừng conversion và chuyển sang phương án Keras retrain.
